In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ealaxi/paysim1/PS_20174392719_1491204439457_log.csv


## Preprocessing and Feature Engineering

In [3]:
import pandas as pd
import numpy as np

# 1. LOAD
df = pd.read_csv("/kaggle/input/datasets/ealaxi/paysim1/PS_20174392719_1491204439457_log.csv")
print("Raw shape:", df.shape)


# 2. FILTER TO WHERE FRAUD ACTUALLY OCCURS
fraud_types = ["TRANSFER", "CASH_OUT"]
df = df[df["type"].isin(fraud_types)].copy()
print("After filtering to TRANSFER/CASH_OUT:", df.shape)
print("Fraud rate now: {:.4%}".format(df["isFraud"].mean()))

# 3. FEATURE ENGINEERING
df["errorBalanceOrig"] = df["newbalanceOrig"] + df["amount"] - df["oldbalanceOrg"]
df["errorBalanceDest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]

# Flag: origin account fully drained to zero
df["origDrainedToZero"] = ((df["oldbalanceOrg"] > 0) & (df["newbalanceOrig"] == 0)).astype(int)

# Flag: destination started empty (common in mule/fraud accounts)
df["destStartedEmpty"] = (df["oldbalanceDest"] == 0).astype(int)

# Encode transaction type (only 2 left, so simple binary)
df["type_TRANSFER"] = (df["type"] == "TRANSFER").astype(int)

# 4. DROP LEAKAGE / NON-PREDICTIVE COLUMNS
drop_cols = ["nameOrig", "nameDest", "isFlaggedFraud", "type"]
df = df.drop(columns=drop_cols)

print("\nFinal columns:", list(df.columns))
print("Final shape:", df.shape)

# 5. SAVE PROCESSED DATA
df.to_parquet("fraud_processed.parquet", index=False)
print("\nSaved -> fraud_processed.parquet")

# Quick sanity peek
print("\n--- Sample ---")
print(df.head())
print("\n--- Fraud vs non-fraud means (key features) ---")
print(df.groupby("isFraud")[["amount", "errorBalanceOrig", "errorBalanceDest",
                              "origDrainedToZero", "destStartedEmpty"]].mean())

Raw shape: (6362620, 11)
After filtering to TRANSFER/CASH_OUT: (2770409, 11)
Fraud rate now: 0.2965%

Final columns: ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'errorBalanceOrig', 'errorBalanceDest', 'origDrainedToZero', 'destStartedEmpty', 'type_TRANSFER']
Final shape: (2770409, 12)

Saved -> fraud_processed.parquet

--- Sample ---
    step     amount  oldbalanceOrg  newbalanceOrig  oldbalanceDest  \
2      1     181.00          181.0             0.0             0.0   
3      1     181.00          181.0             0.0         21182.0   
15     1  229133.94        15325.0             0.0          5083.0   
19     1  215310.30          705.0             0.0         22425.0   
24     1  311685.89        10835.0             0.0          6267.0   

    newbalanceDest  isFraud  errorBalanceOrig  errorBalanceDest  \
2             0.00        1              0.00             181.0   
3             0.00        1              0.00       

## Training and Comparing all Four Models

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             classification_report, confusion_matrix)
import xgboost as xgb
import lightgbm as lgb
import joblib
import warnings
warnings.filterwarnings("ignore")

# 1. LOAD PROCESSED DATA
df = pd.read_parquet("fraud_processed.parquet")

X = df.drop(columns=["isFraud"])
y = df["isFraud"]
feature_names = list(X.columns)
print("Features:", feature_names)
print("Total rows:", len(X), "| Fraud:", y.sum(), "({:.3%})".format(y.mean()))

# 2. TRAIN / TEST SPLIT (stratified to preserve fraud ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
print("\nTrain:", X_train.shape, "| Test:", X_test.shape)
print("Train fraud:", y_train.sum(), "| Test fraud:", y_test.sum())

# Scale features (needed for Logistic Regression; trees don't care but it's harmless)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Class imbalance ratio for the boosting models
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print("scale_pos_weight (neg/pos): {:.1f}".format(scale_pos_weight))

# 3. DEFINE MODELS (all with class-imbalance handling)
models = {
    "LogisticRegression": LogisticRegression(
        class_weight="balanced", max_iter=1000, n_jobs=-1
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced",
        n_jobs=-1, random_state=42
    ),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        scale_pos_weight=scale_pos_weight, eval_metric="aucpr",
        n_jobs=-1, random_state=42
    ),
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1, random_state=42, verbose=-1
    ),
}

# 4. TRAIN + EVALUATE
results = {}
trained = {}

for name, model in models.items():
    print("\n" + "="*60)
    print("Training:", name)
    # LogReg uses scaled data; trees use raw
    if name == "LogisticRegression":
        model.fit(X_train_scaled, y_train)
        proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]

    pr_auc = average_precision_score(y_test, proba)
    roc = roc_auc_score(y_test, proba)
    preds = (proba >= 0.5).astype(int)

    results[name] = {"PR_AUC": pr_auc, "ROC_AUC": roc}
    trained[name] = model

    print(f"PR-AUC: {pr_auc:.4f} | ROC-AUC: {roc:.4f}")
    print(classification_report(y_test, preds, digits=4))

# 5. SUMMARY TABLE
print("\n" + "="*60)
print("MODEL COMPARISON (sorted by PR-AUC)")
summary = pd.DataFrame(results).T.sort_values("PR_AUC", ascending=False)
print(summary.round(4))

# 6. SAVE EVERYTHING FOR NEXT STEPS (SHAP + threshold tuning + app)
best_name = summary.index[0]
joblib.dump(trained[best_name], "best_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(feature_names, "feature_names.pkl")
# Save test set so we don't recompute the split later
joblib.dump((X_test, y_test), "test_data.pkl")
print(f"\nBest model: {best_name} -> saved best_model.pkl")

Features: ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'errorBalanceOrig', 'errorBalanceDest', 'origDrainedToZero', 'destStartedEmpty', 'type_TRANSFER']
Total rows: 2770409 | Fraud: 8213 (0.296%)

Train: (2077806, 11) | Test: (692603, 11)
Train fraud: 6160 | Test fraud: 2053
scale_pos_weight (neg/pos): 336.3

Training: LogisticRegression
PR-AUC: 0.6496 | ROC-AUC: 0.9911
              precision    recall  f1-score   support

           0     0.9999    0.9574    0.9782    690550
           1     0.0629    0.9620    0.1180      2053

    accuracy                         0.9574    692603
   macro avg     0.5314    0.9597    0.5481    692603
weighted avg     0.9971    0.9574    0.9756    692603


Training: RandomForest
PR-AUC: 0.9974 | ROC-AUC: 0.9988
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000    690550
           1     0.9995    0.9966    0.9980      2053

    accuracy                      

### Note on LightGBM and class imbalance
In the initial run, all four models used scale_pos_weight = 336 to counter the 0.3% fraud rate. This worked well for XGBoost and Random Forest but caused LightGBM to fail badly (PR-AUC 0.02). The cause is LightGBM's leaf-wise tree growth: combined with such an extreme positive-class weight, it over-specialised on the minority class and produced poorly ranked probabilities. The fix was to replace the aggressive manual weight with the gentler is_unbalance=True and to constrain leaf growth via min_child_samples and num_leaves, which restored LightGBM to a competitive PR-AUC. This highlights that imbalance-handling settings are not one-size-fits-all across gradient-boosting libraries and must be tuned per algorithm.

In [5]:
import pandas as pd
import numpy as np
import joblib
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             classification_report)
import warnings
warnings.filterwarnings("ignore")

# 1. RELOAD DATA + RECREATE THE EXACT SAME SPLIT
df = pd.read_parquet("fraud_processed.parquet")
X = df.drop(columns=["isFraud"])
y = df["isFraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# 2. WHY IT FAILED + THE FIX
#    Before: scale_pos_weight=336 forced LightGBM's leaf-wise trees to
#    over-favour the positive class so hard that probability ranking broke.
#    Fix: use is_unbalance instead of a brutal manual weight, and constrain
#    leaf growth (num_leaves, min_child_samples) so leaves stay meaningful.
lgbm = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=100,   
    subsample=0.8,
    colsample_bytree=0.8,
    is_unbalance=True,       
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)

print("Training fixed LightGBM...")
lgbm.fit(X_train, y_train)
proba = lgbm.predict_proba(X_test)[:, 1]

pr_auc = average_precision_score(y_test, proba)
roc = roc_auc_score(y_test, proba)
preds = (proba >= 0.5).astype(int)

print(f"\nFixed LightGBM -> PR-AUC: {pr_auc:.4f} | ROC-AUC: {roc:.4f}")
print(classification_report(y_test, preds, digits=4))

# 3. REBUILD THE FULL COMPARISON TABLE
prior = {
    "XGBoost":            {"PR_AUC": 0.9976, "ROC_AUC": 0.9989},
    "RandomForest":       {"PR_AUC": 0.9974, "ROC_AUC": 0.9988},
    "LogisticRegression": {"PR_AUC": 0.6496, "ROC_AUC": 0.9911},
}
prior["LightGBM (fixed)"] = {"PR_AUC": round(pr_auc, 4), "ROC_AUC": round(roc, 4)}

summary = pd.DataFrame(prior).T.sort_values("PR_AUC", ascending=False)
print("\n" + "="*60)
print("FINAL MODEL COMPARISON (sorted by PR-AUC)")
print(summary.round(4))

# Save the fixed model 
joblib.dump(lgbm, "lightgbm_fixed.pkl")
print("\nSaved -> lightgbm_fixed.pkl")

Training fixed LightGBM...

Fixed LightGBM -> PR-AUC: 0.9974 | ROC-AUC: 0.9990
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000    690550
           1     0.9956    0.9966    0.9961      2053

    accuracy                         1.0000    692603
   macro avg     0.9978    0.9983    0.9980    692603
weighted avg     1.0000    1.0000    1.0000    692603


FINAL MODEL COMPARISON (sorted by PR-AUC)
                    PR_AUC  ROC_AUC
XGBoost             0.9976   0.9989
RandomForest        0.9974   0.9988
LightGBM (fixed)    0.9974   0.9990
LogisticRegression  0.6496   0.9911

Saved -> lightgbm_fixed.pkl


## SHAP Explainability plus Cost-Aware Threshold Tuning

In [6]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, f1_score, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

# 1. LOAD SAVED MODEL + TEST DATA
model = joblib.load("best_model.pkl")          # XGBoost
feature_names = joblib.load("feature_names.pkl")
X_test, y_test = joblib.load("test_data.pkl")

proba = model.predict_proba(X_test)[:, 1]
print("Loaded XGBoost + test set:", X_test.shape)

# PART A — SHAP EXPLAINABILITY (auditable, justifiable flags)
print("\nComputing SHAP values (sampling 5k rows for speed)...")
# TreeExplainer is exact + fast for tree models; sample for the plots
sample_idx = np.random.RandomState(42).choice(len(X_test), 5000, replace=False)
X_sample = X_test.iloc[sample_idx]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

# Global feature importance (mean |SHAP|)
importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)
print("\n--- GLOBAL FEATURE IMPORTANCE (SHAP) ---")
print(importance.to_string(index=False))

# Beeswarm summary plot
plt.figure()
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig("shap_beeswarm.png", dpi=130, bbox_inches="tight")
plt.close()
print("\nSaved -> shap_beeswarm.png")

# PART B — COST-AWARE THRESHOLD TUNING
COST_FN = 100   # cost of letting fraud through (relative units)
COST_FP = 1     # cost of investigating a false alarm

thresholds = np.linspace(0.01, 0.99, 99)
costs, f1s = [], []
for t in thresholds:
    preds = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    total_cost = fn * COST_FN + fp * COST_FP
    costs.append(total_cost)
    f1s.append(f1_score(y_test, preds))

costs = np.array(costs)
best_cost_idx = costs.argmin()
best_f1_idx = int(np.argmax(f1s))

print("\n--- THRESHOLD ANALYSIS ---")
print(f"Default threshold 0.50:")
preds50 = (proba >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, preds50).ravel()
print(f"  FP={fp}  FN={fn}  F1={f1_score(y_test, preds50):.4f}  cost={fn*COST_FN+fp*COST_FP}")

t_cost = thresholds[best_cost_idx]
preds_c = (proba >= t_cost).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, preds_c).ravel()
print(f"\nCost-optimal threshold {t_cost:.2f} (COST_FN={COST_FN}, COST_FP={COST_FP}):")
print(f"  FP={fp}  FN={fn}  F1={f1_score(y_test, preds_c):.4f}  cost={fn*COST_FN+fp*COST_FP}")

t_f1 = thresholds[best_f1_idx]
print(f"\nF1-optimal threshold {t_f1:.2f}: F1={f1s[best_f1_idx]:.4f}")

# Plot cost vs threshold
plt.figure(figsize=(7,4))
plt.plot(thresholds, costs)
plt.axvline(t_cost, color="red", linestyle="--", label=f"cost-optimal {t_cost:.2f}")
plt.axvline(0.5, color="grey", linestyle=":", label="default 0.50")
plt.xlabel("Decision threshold"); plt.ylabel("Total expected cost")
plt.title(f"Cost-aware threshold (FN={COST_FN}x, FP={COST_FP}x)")
plt.legend(); plt.tight_layout()
plt.savefig("threshold_cost.png", dpi=130)
plt.close()
print("\nSaved -> threshold_cost.png")

# Save chosen threshold for the app
joblib.dump({"threshold": float(t_cost), "cost_fn": COST_FN, "cost_fp": COST_FP},
            "decision_config.pkl")
print(f"\nSaved decision threshold {t_cost:.2f} -> decision_config.pkl")

Loaded XGBoost + test set: (692603, 11)

Computing SHAP values (sampling 5k rows for speed)...

--- GLOBAL FEATURE IMPORTANCE (SHAP) ---
          feature  mean_abs_shap
 errorBalanceOrig       8.428262
             step       2.136231
   oldbalanceDest       1.794037
   newbalanceOrig       1.170470
origDrainedToZero       1.117630
   newbalanceDest       1.009771
           amount       0.819576
    oldbalanceOrg       0.537072
 errorBalanceDest       0.472719
    type_TRANSFER       0.204669
 destStartedEmpty       0.088407

Saved -> shap_beeswarm.png

--- THRESHOLD ANALYSIS ---
Default threshold 0.50:
  FP=7  FN=7  F1=0.9966  cost=707

Cost-optimal threshold 0.11 (COST_FN=100, COST_FP=1):
  FP=14  FN=6  F1=0.9951  cost=614

F1-optimal threshold 0.99: F1=0.9978

Saved -> threshold_cost.png

Saved decision threshold 0.11 -> decision_config.pkl


## Rules-vs-ML baseline + Calibration

In [7]:
import pandas as pd
import numpy as np
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, brier_score_loss,
                             precision_score, recall_score, f1_score)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
import warnings
warnings.filterwarnings("ignore")

# LOAD model + test set
model = joblib.load("best_model.pkl")
X_test, y_test = joblib.load("test_data.pkl")
proba = model.predict_proba(X_test)[:, 1]
preds = (proba >= 0.5).astype(int)

# PART A — RULES vs ML
rule_preds = ((X_test["type_TRANSFER"] == 1) & (X_test["amount"] > 200000)).astype(int)

def scorecard(name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "model": name,
        "frauds_caught": int(tp),
        "frauds_missed": int(fn),
        "recall": recall_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

comparison = pd.DataFrame([
    scorecard("Rule-based (amount>200k)", y_test, rule_preds),
    scorecard("ML (XGBoost @0.50)", y_test, preds),
])
print("="*65)
print("RULES vs ML  (test set: {} frauds)".format(int(y_test.sum())))
print("="*65)
print(comparison.to_string(index=False))

# PART B — CALIBRATION
brier_raw = brier_score_loss(y_test, proba)
frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=10, strategy="quantile")

# Platt-scale (sigmoid) calibration via CV on the model's outputs
calibrated = CalibratedClassifierCV(model, method="sigmoid", cv="prefit")
calibrated.fit(X_test, y_test)
proba_cal = calibrated.predict_proba(X_test)[:, 1]
brier_cal = brier_score_loss(y_test, proba_cal)
frac_pos_c, mean_pred_c = calibration_curve(y_test, proba_cal, n_bins=10, strategy="quantile")

print("\n" + "="*65)
print("CALIBRATION")
print("="*65)
print(f"Brier score (raw):        {brier_raw:.5f}")
print(f"Brier score (calibrated): {brier_cal:.5f}")
improvement = (brier_raw - brier_cal) / brier_raw * 100
print(f"Improvement: {improvement:.1f}%")

# Reliability plot
plt.figure(figsize=(6,6))
plt.plot([0,1],[0,1],"k:",label="Perfect")
plt.plot(mean_pred, frac_pos, "o-", label=f"Raw (Brier {brier_raw:.4f})")
plt.plot(mean_pred_c, frac_pos_c, "s-", label=f"Calibrated (Brier {brier_cal:.4f})")
plt.xlabel("Mean predicted probability"); plt.ylabel("Observed fraud fraction")
plt.title("Reliability curve"); plt.legend(); plt.tight_layout()
plt.savefig("calibration_curve.png", dpi=130)
plt.close()
print("\nSaved -> calibration_curve.png")

RULES vs ML  (test set: 2053 frauds)
                   model  frauds_caught  frauds_missed   recall  precision       f1
Rule-based (amount>200k)            666           1387 0.324403   0.006505 0.012754
      ML (XGBoost @0.50)           2046              7 0.996590   0.996590 0.996590

CALIBRATION
Brier score (raw):        0.00002
Brier score (calibrated): 0.00002
Improvement: 5.1%

Saved -> calibration_curve.png


##  Temporal Split

In [8]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             classification_report, confusion_matrix)
import warnings
warnings.filterwarnings("ignore")

# LOAD processed data
df = pd.read_parquet("fraud_processed.parquet")

# TEMPORAL SPLIT: train on the past, test on the future.
cutoff = df["step"].quantile(0.75)
print(f"Time cutoff at step = {cutoff:.0f} (of max {df['step'].max()})")

train = df[df["step"] <= cutoff]
test  = df[df["step"] >  cutoff]

X_train = train.drop(columns=["isFraud"]); y_train = train["isFraud"]
X_test  = test.drop(columns=["isFraud"]);  y_test  = test["isFraud"]

print(f"Train: {X_train.shape} | frauds: {y_train.sum()} ({y_train.mean():.3%})")
print(f"Test:  {X_test.shape} | frauds: {y_test.sum()} ({y_test.mean():.3%})")

# Train XGBoost (same config as the winning model)
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
spw = neg / pos

model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    scale_pos_weight=spw, eval_metric="aucpr",
    n_jobs=-1, random_state=42
)
print("\nTraining XGBoost on temporal split...")
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
preds = (proba >= 0.5).astype(int)

pr_auc = average_precision_score(y_test, proba)
roc = roc_auc_score(y_test, proba)

print("\n" + "="*60)
print("TEMPORAL SPLIT RESULTS (train past -> test future)")
print("="*60)
print(f"PR-AUC: {pr_auc:.4f} | ROC-AUC: {roc:.4f}")
print(classification_report(y_test, preds, digits=4))

tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
print(f"Caught {tp}/{tp+fn} future frauds | False alarms: {fp}")

print("\n--- COMPARISON ---")
print(f"Random split PR-AUC:   0.9976")
print(f"Temporal split PR-AUC: {pr_auc:.4f}")
print("If temporal is close to random, the model generalises across time (no leakage).")

Time cutoff at step = 332 (of max 743)
Train: (2078818, 11) | frauds: 3733 (0.180%)
Test:  (691591, 11) | frauds: 4480 (0.648%)

Training XGBoost on temporal split...

TEMPORAL SPLIT RESULTS (train past -> test future)
PR-AUC: 1.0000 | ROC-AUC: 1.0000
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000    687111
           1     1.0000    0.9998    0.9999      4480

    accuracy                         1.0000    691591
   macro avg     1.0000    0.9999    0.9999    691591
weighted avg     1.0000    1.0000    1.0000    691591

Caught 4479/4480 future frauds | False alarms: 0

--- COMPARISON ---
Random split PR-AUC:   0.9976
Temporal split PR-AUC: 1.0000
If temporal is close to random, the model generalises across time (no leakage).
